## Imports


In [1]:
import numpy as np
import pandas as pd
import os

## Show experiment config

In [18]:
# DuckDB nutzen, um die Daten zu aggregieren
con = duckdb.connect()

query = f"""

    SELECT 
        "experiment"."config" AS config, 
    FROM read_parquet('{parquet_files}')
    LIMIT(1) 
--TODO: change file write so that config (world) is only in first parquet file
"""

result = con.execute(query).df()

result

,config
0,"world=WorldConfig(grid_rows=16, grid_cols=16) agent=AgentConfig(type=<AgentType.SARSA: 'sarsa'>, alpha=0.05, gamma=0.99, epsilon=0.5, epsilon_decay=0.9995, epsilon_min=0.01) training=TrainingConfig(num_episodes=10000, max_steps_per_episode=100, validate_interval=100) rewards=RewardsConfig(default=-1, invalid=-100, wall=-10) files=FilesConfig(experiment_dir='experiments', board='board.npy', output_prefix='board_with_path', q_table_prefix='q_table', training_path_prefix='training_path_and_rewards')"


In [25]:
import duckdb
from IPython.display import Markdown, display


# Query all experiments
results = duckdb.query("""
    SELECT 
        experiment.episode.nr as episode_number,
        experiment.episode.mode as mode,
        len(experiment.episode.steps) as num_steps,
        experiment.episode.steps[1].reward_to_go as total_reward
    FROM read_parquet('experiments/experiment_*.parquet')
    WHERE mode = 'validate'
    ORDER BY episode_number
""").df()

# Note: reward_to_go is only accumulated in validation epochs
display(Markdown("# Reward in Episode"))
pd.set_option('display.max_rows', None)
results

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

# Reward in Episode

,episode_number,mode,num_steps,total_reward
0,100,validate,100,-100.0
1,200,validate,100,-100.0
2,300,validate,100,-100.0
3,400,validate,100,-100.0
4,500,validate,100,-100.0
5,600,validate,100,-100.0
6,700,validate,100,-100.0
7,800,validate,100,-100.0
8,900,validate,100,-100.0
9,1000,validate,100,-100.0


# Find all steps where agent runs against a wall and if it was due to epsilon action


In [26]:
import duckdb

# Pfad zu deinen Dateien (nutzt Wildcards für alle Episoden)
parquet_files = "./experiments/experiment_*.parquet"

# DuckDB nutzen, um die Daten zu aggregieren
con = duckdb.connect()

query = f"""
SELECT * FROM(
    SELECT 
        "experiment"."episode"."nr" AS episode_number, 
        "experiment"."episode"."mode" AS episode_mode,  
        step.reward AS reward, 
        -- step.strategy AS strategy,
        COUNT(*) AS count
    FROM read_parquet('{parquet_files}')
    CROSS JOIN UNNEST("experiment"."episode"."steps") AS t(step) 
    GROUP BY episode_number, episode_mode, reward
    ORDER BY episode_number, episode_mode, reward)
    WHERE reward=-10
"""

result = con.execute(query).df()

result

# Optional: Als eine einzige CSV oder Parquet zusammenfassen
# con.execute(f"COPY ({query}) TO 'aggregated_rewards.csv' (HEADER, DELIMITER ',')")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,episode_number,episode_mode,reward,count
0,1,train,-10.0,9
1,2,train,-10.0,19
2,3,train,-10.0,12
3,4,train,-10.0,7
4,5,train,-10.0,5
5,6,train,-10.0,6
6,7,train,-10.0,19
7,8,train,-10.0,8
8,9,train,-10.0,12
9,10,train,-10.0,7


In [17]:
# DuckDB nutzen, um die Daten zu aggregieren
con = duckdb.connect()

query = f"""

    SELECT 
        "experiment"."config" AS config, 
    FROM read_parquet('{parquet_files}')
    LIMIT(1) 
--TODO: change file write so that config (world) is only in first parquet file
"""

result = con.execute(query).df()

result

,config
0,"world=WorldConfig(grid_rows=16, grid_cols=16) agent=AgentConfig(type=<AgentType.SARSA: 'sarsa'>, alpha=0.05, gamma=0.99, epsilon=0.5, epsilon_decay=0.9995, epsilon_min=0.01) training=TrainingConfig(num_episodes=10000, max_steps_per_episode=100, validate_interval=100) rewards=RewardsConfig(default=-1, invalid=-100, wall=-10) files=FilesConfig(experiment_dir='experiments', board='board.npy', output_prefix='board_with_path', q_table_prefix='q_table', training_path_prefix='training_path_and_rewards')"
